# BondSpot dashboard — duration & debt composition over time

Wykresy:
1. **Portfolio-weighted metryki** (Mod/Mac Duration, ATM, ATR) w czasie
2. **Per typ obligacji** (DS/PS/WS/WZ/IZ/...) — facet 2×2
3. **Skład długu** — stacked area (bondy + bony skarbowe)

## Setup
1. `pip install -r ../requirements-notebook.txt`
2. Utwórz `.env` w roocie repo z `SUPABASE_URL` i `SUPABASE_SERVICE_ROLE_KEY`
3. Run all cells

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from plotly.subplots import make_subplots

# Renderer wymagany zeby fig.show() emitowal HTML/JS przy nbconvert -> HTML;
# live Jupyter zwykle uzywa 'plotly_mimetype' ale to nie renderuje w GH Pages.
pio.renderers.default = "notebook_connected"

load_dotenv(Path("..") / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000  # Supabase PGRST_DB_MAX_ROWS default - kapuje response do 1000


def _paginate(method, url, *, json_body=None, timeout=120, max_pages=200):
    """Stronicowane wywolanie endpoint-u Supabase. PostgREST tnie response do
    PGRST_DB_MAX_ROWS (1000 default na Supabase) niezaleznie od Range header,
    wiec idziemy w petli inkrementujac offset i konkatenujac wyniki."""
    rows: list = []
    for page in range(max_pages):
        offset = page * PAGE_SIZE
        h = {**HEADERS,
             "Range-Unit": "items",
             "Range": f"{offset}-{offset + PAGE_SIZE - 1}"}
        kw = {"headers": h, "timeout": timeout}
        if json_body is not None:
            kw["json"] = json_body
        r = requests.request(method, url, **kw)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        if not isinstance(chunk, list):
            return chunk  # scalar / dict response - return as-is
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            return rows
    raise RuntimeError(f"_paginate hit max_pages={max_pages} without finishing")


def rpc(name: str, payload: dict | None = None) -> pd.DataFrame:
    rows = _paginate("POST", f"{SUPABASE_URL}/rest/v1/rpc/{name}",
                     json_body=payload or {})
    return pd.DataFrame(rows)


def fetch_view(name: str, query: str = "?select=*") -> pd.DataFrame:
    rows = _paginate("GET", f"{SUPABASE_URL}/rest/v1/{name}{query}")
    return pd.DataFrame(rows)


print("Connected.")

## 1. Portfolio-weighted metryki w czasie

Z `v_portfolio_metrics_daily` (ważone outstanding-em dziennym, bondy hurtowe).

In [ ]:
df1 = fetch_view("v_portfolio_metrics_daily", "?select=*&order=fixing_date.asc")
df1["fixing_date"] = pd.to_datetime(df1["fixing_date"])
for c in ["portfolio_mod_duration", "portfolio_mac_duration", "portfolio_atm",
         "portfolio_atr", "portfolio_yield_pct", "total_outstanding_mln_pln"]:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

print(f"Days: {len(df1)},  range: {df1.fixing_date.min().date()} → {df1.fixing_date.max().date()}")
df1.tail()

In [ ]:
fig = go.Figure()
for col, label in [
    ("portfolio_mod_duration", "Modified Duration"),
    ("portfolio_mac_duration", "Macaulay Duration"),
    ("portfolio_atm", "ATM (years to maturity)"),
    ("portfolio_atr", "ATR (years to refixing)"),
]:
    fig.add_trace(go.Scatter(x=df1["fixing_date"], y=df1[col], name=label, mode="lines"))

fig.update_layout(
    title="Portfolio-weighted metryki polskiego długu (bondy hurtowe)",
    xaxis_title="Data fixingu (EOD = sesja 2)",
    yaxis_title="Lata",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 2. Per typ obligacji (DS, PS, WS, WZ, IZ, …)

Każdy z 4 paneli pokazuje jedną metrykę, kolory = typy obligacji.

Czego się spodziewać:
- **WS** (długie 20-30Y) — najwyższe Mac/Mod Duration
- **WZ** (floatery) — Mod/Mac ≈ ATR (≤ 0.5Y), bo resetują się co 6mc
- **OK** — wszystkie metryki = ATM (zerokuponowe), maleją liniowo do wykupu

In [ ]:
df2 = fetch_view("v_portfolio_metrics_by_type", "?select=*&order=fixing_date.asc,bond_type.asc")
df2["fixing_date"] = pd.to_datetime(df2["fixing_date"])
for c in ["total_mln_pln", "w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]:
    df2[c] = pd.to_numeric(df2[c], errors="coerce")

print(f"Rows: {len(df2)},  types: {sorted(df2.bond_type.unique())}")
df2.tail()

In [ ]:
metrics = [
    ("w_mod_duration", "Modified Duration (lata)"),
    ("w_mac_duration", "Macaulay Duration (lata)"),
    ("w_atm", "ATM (lata)"),
    ("w_atr", "ATR (lata)"),
]
fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

types_sorted = sorted(df2.bond_type.unique())
colors = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
color_map = {bt: colors[i % len(colors)] for i, bt in enumerate(types_sorted)}

for i, (col, _) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    for bt in types_sorted:
        sub = df2[df2.bond_type == bt].sort_values("fixing_date")
        fig.add_trace(
            go.Scatter(
                x=sub["fixing_date"], y=sub[col],
                name=bt, legendgroup=bt,
                showlegend=(i == 0), mode="lines",
                line=dict(color=color_map[bt], width=1.5),
            ),
            row=row, col=c,
        )

fig.update_layout(
    title="Metryki ważone outstanding per typ obligacji",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.05),
)
fig.show()

## 3. Skład długu — stacked area

Bondy hurtowe per typ + bony skarbowe (jako jeden kubełek `tbill`). Pokazuje jak ewoluowała struktura zadłużenia.

Resample do miesięcznych snapshotów (end-of-month), inaczej wykres jest zaszumiony przy 3000+ dni × 8 typów.

In [ ]:
# Bondy: jeden GET na widok bondow per (data, typ)
df3a = fetch_view("v_debt_composition_bonds", "?select=*&order=fixing_date.asc,bond_type.asc")
df3a["fixing_date"] = pd.to_datetime(df3a["fixing_date"])
df3a["outstanding_mln_pln"] = pd.to_numeric(df3a["outstanding_mln_pln"], errors="coerce")

# T-bille: raw change-points z tbill_outstanding, forward-fill do dziennych w pandasie.
# (Robione w pandasie zamiast widoku, bo LATERAL + tbills_outstanding_at w widoku PostgREST
# zwracal 500 - patrz komentarz w dashboard_rpcs.sql.)
tbill_raw = fetch_view(
    "tbill_outstanding",
    "?select=isin,change_date,balance_mln_pln&order=isin.asc,change_date.asc",
)
tbill_raw["change_date"] = pd.to_datetime(tbill_raw["change_date"])
tbill_raw["balance_mln_pln"] = pd.to_numeric(tbill_raw["balance_mln_pln"], errors="coerce")

# Pivot do wide (kolumny = ISIN), forward-fill (balance trwa miedzy change pointami)
tbill_wide = (
    tbill_raw.pivot_table(index="change_date", columns="isin",
                          values="balance_mln_pln", aggfunc="last")
)
tbill_wide = tbill_wide.resample("D").ffill().fillna(0)
tbill_daily = tbill_wide.sum(axis=1)
# Trzymamy tylko dni gdzie outstanding > 0 (przed pierwsza emisja t-bili nic nie pokazujemy)
tbill_daily = tbill_daily[tbill_daily > 0]

tbill_df = (
    tbill_daily.reset_index()
    .rename(columns={"change_date": "fixing_date", 0: "outstanding_mln_pln"})
)
tbill_df.columns = ["fixing_date", "outstanding_mln_pln"]
tbill_df["bond_type"] = "tbill"

# Polacz bondy + tbill
df3 = pd.concat([df3a, tbill_df[df3a.columns]], ignore_index=True)

# Pivot do stacked area: index=date, cols=type
piv = df3.pivot_table(index="fixing_date", columns="bond_type",
                     values="outstanding_mln_pln", aggfunc="sum")
piv = piv.fillna(0).resample("ME").last()

# Largest types first on stack
order = piv.mean().sort_values(ascending=False).index.tolist()
piv = piv[order]

print(f"Months: {len(piv)},  types: {order}")
piv.tail()

In [ ]:
fig = go.Figure()
for bt in order:
    fig.add_trace(go.Scatter(
        x=piv.index, y=piv[bt] / 1000.0,
        name=bt, mode="lines", stackgroup="one",
        hovertemplate="%{y:.1f} bln PLN<extra>" + bt + "</extra>",
    ))

fig.update_layout(
    title="Skład długu skarbowego (bondy hurtowe + bony) — monthly snapshot",
    xaxis_title="Data",
    yaxis_title="Outstanding (bln PLN)",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

---

**Tip:** wykresy plotly są interaktywne — najedź myszą żeby zobaczyć wartości, zaznacz prostokąt żeby przybliżyć, podwójny klik żeby zresetować. Trace w legendzie można wyłączać klikiem.